# Python Challenge 2 – CLI Contact Book (CSV-Powered)
A terminal contact book stored in `contacts.csv` with columns **Name, Phone, Email**.

Menu: **Add Contact · View All Contacts · Search Contact · Exit**

Requirements covered
* file `contacts.csv` is created automatically if missing
* clean menu interface

Bonus covered
* table-like listing · partial (substring) search, case-insensitive · duplicate names are blocked

In [1]:
import csv
import os

CSV_FILE = "contacts.csv"
FIELDS = ["Name", "Phone", "Email"]

## File handling
`ensure_file()` creates the CSV with a header row if it does not exist yet.

In [2]:
def ensure_file(path=CSV_FILE):
    if not os.path.exists(path):
        with open(path, "w", newline="", encoding="utf-8") as f:
            csv.DictWriter(f, fieldnames=FIELDS).writeheader()

def load_contacts(path=CSV_FILE):
    ensure_file(path)
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

## Core operations

In [3]:
def add_contact(name, phone, email, path=CSV_FILE):
    name, phone, email = name.strip(), phone.strip(), email.strip()
    if not name:
        return False, "Name cannot be empty."
    if any(c["Name"].lower() == name.lower() for c in load_contacts(path)):
        return False, f"A contact named '{name}' already exists."
    with open(path, "a", newline="", encoding="utf-8") as f:
        csv.DictWriter(f, fieldnames=FIELDS).writerow({"Name": name, "Phone": phone, "Email": email})
    return True, f"Contact '{name}' saved."

def search_contacts(query, path=CSV_FILE):
    """Partial, case-insensitive match on the name."""
    q = query.strip().lower()
    return [c for c in load_contacts(path) if q in c["Name"].lower()]

def print_table(contacts):
    if not contacts:
        print("  (no contacts to show)")
        return
    w_name = max(len("Name"), *(len(c["Name"]) for c in contacts))
    w_phone = max(len("Phone"), *(len(c["Phone"]) for c in contacts))
    w_mail = max(len("Email"), *(len(c["Email"]) for c in contacts))
    line = f"+-{'-' * w_name}-+-{'-' * w_phone}-+-{'-' * w_mail}-+"
    print(line)
    print(f"| {'Name':<{w_name}} | {'Phone':<{w_phone}} | {'Email':<{w_mail}} |")
    print(line)
    for c in contacts:
        print(f"| {c['Name']:<{w_name}} | {c['Phone']:<{w_phone}} | {c['Email']:<{w_mail}} |")
    print(line)

## Menu loop

In [4]:
def main(path=CSV_FILE):
    ensure_file(path)
    while True:
        print("\n===== CONTACT BOOK =====")
        print("1. Add Contact")
        print("2. View All Contacts")
        print("3. Search Contact")
        print("4. Exit")
        choice = input("Choose an option (1-4): ").strip()

        if choice == "1":
            name = input("Name : ")
            phone = input("Phone: ")
            email = input("Email: ")
            print(add_contact(name, phone, email, path)[1])
        elif choice == "2":
            print_table(load_contacts(path))
        elif choice == "3":
            q = input("Search name: ")
            results = search_contacts(q, path)
            print(f"{len(results)} match(es) for '{q}':")
            print_table(results)
        elif choice == "4":
            print("Goodbye!")
            break
        else:
            print("Invalid option, please choose 1-4.")

## Demo (scripted input)
Uses a throw-away file `demo_contacts.csv`, so your real `contacts.csv` is never touched. The script adds three contacts, tries a duplicate, lists them, and does a partial search for "ra".

In [5]:
from unittest.mock import patch

def run_with_inputs(func, inputs, *args):
    it = iter(inputs)
    def fake_input(prompt=""):
        value = next(it)
        print(f"{prompt}{value}")
        return value
    with patch("builtins.input", fake_input):
        return func(*args)

if os.path.exists("demo_contacts.csv"):
    os.remove("demo_contacts.csv")

run_with_inputs(main, [
    "1", "Asha Sharma", "9876543210", "asha@example.com",
    "1", "Ravi Kumar",  "9123456780", "ravi@example.com",
    "1", "Anand Rao",   "9988776655", "anand@example.com",
    "1", "asha sharma", "0000000000", "dup@example.com",   # duplicate
    "2",
    "3", "ra",                                              # partial match -> Ravi Kumar, Anand Rao
    "9",                                                    # invalid option
    "4",
], "demo_contacts.csv")


===== CONTACT BOOK =====
1. Add Contact
2. View All Contacts
3. Search Contact
4. Exit
Choose an option (1-4): 1
Name : Asha Sharma
Phone: 9876543210
Email: asha@example.com
Contact 'Asha Sharma' saved.

===== CONTACT BOOK =====
1. Add Contact
2. View All Contacts
3. Search Contact
4. Exit
Choose an option (1-4): 1
Name : Ravi Kumar
Phone: 9123456780
Email: ravi@example.com
Contact 'Ravi Kumar' saved.

===== CONTACT BOOK =====
1. Add Contact
2. View All Contacts
3. Search Contact
4. Exit
Choose an option (1-4): 1
Name : Anand Rao
Phone: 9988776655
Email: anand@example.com
Contact 'Anand Rao' saved.

===== CONTACT BOOK =====
1. Add Contact
2. View All Contacts
3. Search Contact
4. Exit
Choose an option (1-4): 1
Name : asha sharma
Phone: 0000000000
Email: dup@example.com
A contact named 'asha sharma' already exists.

===== CONTACT BOOK =====
1. Add Contact
2. View All Contacts
3. Search Contact
4. Exit
Choose an option (1-4): 2
+-------------+------------+-------------------+
| Name    

### Raw CSV file created by the program

In [6]:
print(open("demo_contacts.csv").read())

Name,Phone,Email
Asha Sharma,9876543210,asha@example.com
Ravi Kumar,9123456780,ravi@example.com
Anand Rao,9988776655,anand@example.com



### Clean up the demo file

In [7]:
os.remove("demo_contacts.csv")
print("demo_contacts.csv removed")

demo_contacts.csv removed


## Run it yourself (uses the real `contacts.csv`)

In [8]:
# main()